# 02 · Feature Engineering & Selection (5-Model Pipeline)
Builds leakage-safe derived features (train-only statistics, causal rolling
windows, interactions, demographic ratios), prunes redundant features, and
freezes the final feature contract (`feature_columns.json` + scaler) that the
model trainer and the production backend will both use.

The frozen feature set is shared across all 5 models: Logistic Regression,
Random Forest, XGBoost, LightGBM, and Linear Regression.


In [1]:
import pandas as pd
import numpy as np
import json, joblib
from pathlib import Path
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path("../../data_pipeline/data/processed")
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(DATA_DIR / "train_matrix.csv")
test  = pd.read_csv(DATA_DIR / "test_matrix.csv")

# Defensive alias mapping (protects against upstream naming drift)
ALIASES = {"temperature_mean": "temp_mean_c",
           "rainfall_sum": "rainfall_mm",
           "soil_moisture_mean": "soil_moisture"}
for df in (train, test):
    df.rename(columns={k: v for k, v in ALIASES.items() if k in df.columns}, inplace=True)

print(f"Train: {train.shape} | Test: {test.shape}")
print(f"Risk ratio — train: {train['target_risk'].mean():.2%} | test: {test['target_risk'].mean():.2%}")

Train: (2115, 27) | Test: (893, 27)
Risk ratio — train: 24.44% | test: 7.84%


In [2]:
# 1) Causal 3-month rainfall accumulation (uses only current + past months → no leakage)
full = pd.concat([train.assign(_split="train"), test.assign(_split="test")])
full = full.sort_values(["county", "month"])
full["rain_3m_sum"] = (full.groupby("county")["rainfall_mm"]
                           .transform(lambda s: s.rolling(3, min_periods=1).sum()))
train = full[full._split == "train"].drop(columns="_split").reset_index(drop=True)
test  = full[full._split == "test"].drop(columns="_split").reset_index(drop=True)

# 2) Interaction features (compound stress signals)
for df in (train, test):
    df["heat_moisture_stress"] = df["temp_mean_c"] * df["soil_moisture_deficit"]
    df["rain_ndvi_sync"]       = df["rainfall_anomaly"] * df["ndvi_anomaly"]

# 3) Demographic vulnerability ratios (static county character, population-size independent)
for df in (train, test):
    df["livestock_share"]  = df["livestock_production"] / (df["crop_production"] + df["livestock_production"] + 1)
    df["irrigation_ratio"] = df["irrigation"] / (df["farming"] + 1)
    df["farming_ratio"]    = df["farming"] / (df["total"] + 1)

print("✅ Temporal, interaction and demographic features engineered.")

✅ Temporal, interaction and demographic features engineered.


In [4]:
# County baselines computed STRICTLY on the training period, then applied to both sets
cmean = train.groupby("county")["ndvi_mean"].mean()
cstd  = train.groupby("county")["ndvi_mean"].std().replace(0, 1e-6)

for df in (train, test):
    df["ndvi_zscore"] = (df["ndvi_mean"] - df["county"].map(cmean)) / df["county"].map(cstd)

# Persist for production parity (backend must use the SAME baselines at inference)
pd.DataFrame({"county": cmean.index,
              "ndvi_mean_train": cmean.values,
              "ndvi_std_train": cstd.values}).to_csv(MODEL_DIR / "county_ndvi_stats.csv", index=False)

print("✅ Train-only county baselines saved to models/county_ndvi_stats.csv")

✅ Train-only county baselines saved to models/county_ndvi_stats.csv


In [5]:
CANDIDATES = [
    "month_sin", "month_cos",                                   # seasonality
    "ndvi_lag1", "ndvi_lag3", "ndvi_roll3_mean", "ndvi_roll3_std",
    "ndvi_anomaly", "ndvi_mom_change", "ndvi_zscore",           # vegetation dynamics
    "temp_mean_c", "rainfall_mm", "soil_moisture",             # raw weather
    "rainfall_anomaly", "soil_moisture_deficit", "rain_3m_sum",# engineered weather
    "heat_moisture_stress", "rain_ndvi_sync",                  # interactions
    "livestock_share", "irrigation_ratio", "farming_ratio",    # vulnerability
]
cand = [c for c in CANDIDATES if c in train.columns]

# a) drop near-constant features
stds = train[cand].std()
keep = stds[stds > 1e-6].index.tolist()
print(f"Dropped near-constant: {sorted(set(cand) - set(keep))}")

# b) drop one side of any |corr| > 0.95 pair
corr  = train[keep].corr().abs()
upper = corr.where(np.triu(np.ones_like(corr, dtype=bool), k=1))
drop_set = set()
for col in upper.columns:
    if (upper[col] > 0.95).any():
        drop_set.add(col)
FEATURE_COLS = [c for c in keep if c not in drop_set]
print(f"Dropped highly correlated: {sorted(drop_set)}")

# c) ranking vs target (for the report's feature-analysis section)
rank = train[FEATURE_COLS].corrwith(train["target_risk"]).abs().sort_values(ascending=False)
print("\nFeature relevance (|corr| with target):")
print(rank.round(3))

Dropped near-constant: []
Dropped highly correlated: ['heat_moisture_stress', 'soil_moisture_deficit']

Feature relevance (|corr| with target):
ndvi_zscore         0.711
ndvi_anomaly        0.376
rain_3m_sum         0.278
ndvi_mom_change     0.255
ndvi_lag1           0.219
soil_moisture       0.208
rainfall_mm         0.200
rainfall_anomaly    0.135
ndvi_roll3_mean     0.113
month_cos           0.070
rain_ndvi_sync      0.068
temp_mean_c         0.036
ndvi_roll3_std      0.027
ndvi_lag3           0.022
month_sin           0.014
farming_ratio       0.000
irrigation_ratio    0.000
livestock_share     0.000
dtype: float64


In [6]:
X_train, y_train = train[FEATURE_COLS], train["target_risk"].astype(int)
X_test,  y_test  = test[FEATURE_COLS],  test["target_risk"].astype(int)

# Scaler fitted on TRAIN ONLY (for the logistic-regression baseline in 05)
scaler = StandardScaler().fit(X_train)

# Persist everything the trainer and the backend need
X_train.to_csv(DATA_DIR / "X_train.csv", index=False)
X_test.to_csv(DATA_DIR / "X_test.csv", index=False)
y_train.to_csv(DATA_DIR / "y_train.csv", index=False)
y_test.to_csv(DATA_DIR / "y_test.csv", index=False)
json.dump(FEATURE_COLS, open(MODEL_DIR / "feature_columns.json", "w"), indent=2)
joblib.dump(scaler, MODEL_DIR / "scaler.joblib")

print(f"\n💾 Final feature set: {len(FEATURE_COLS)} features")
print(FEATURE_COLS)
print(f"\nX_train: {X_train.shape} | X_test: {X_test.shape}")
print("✅ Artifacts saved: X/y CSVs, feature_columns.json, scaler.joblib, county_ndvi_stats.csv")


💾 Final feature set: 18 features
['month_sin', 'month_cos', 'ndvi_lag1', 'ndvi_lag3', 'ndvi_roll3_mean', 'ndvi_roll3_std', 'ndvi_anomaly', 'ndvi_mom_change', 'ndvi_zscore', 'temp_mean_c', 'rainfall_mm', 'soil_moisture', 'rainfall_anomaly', 'rain_3m_sum', 'rain_ndvi_sync', 'livestock_share', 'irrigation_ratio', 'farming_ratio']

X_train: (2115, 18) | X_test: (893, 18)
✅ Artifacts saved: X/y CSVs, feature_columns.json, scaler.joblib, county_ndvi_stats.csv


## Feature Catalog (for the report)
| Group | Features | Rationale |
|---|---|---|
| Seasonality | month_sin, month_cos | Cyclical rainfall/growing seasons |
| Vegetation dynamics | ndvi_lag1/3, roll3_mean/std, anomaly, mom_change, zscore | Drought memory & momentum |
| Weather | temp_mean_c, rainfall_mm, soil_moisture, rainfall_anomaly, soil_moisture_deficit, rain_3m_sum | Primary agro-climatic drivers |
| Interactions | heat_moisture_stress, rain_ndvi_sync | Compound stress signals |
| Vulnerability | livestock_share, irrigation_ratio, farming_ratio | County exposure & buffering capacity |